# Voronoi-A* and finite-segment key points

This notebook calls the same implementation as training and evaluation.
Grid coordinates are (row, column) cell centres. Safety uses physical cell
sizes, the robot radius and an uncertainty margin. Historical notebook code
is archived in experiments/results/alignment_20261021_before/.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from global_planner import voronoi_astar_path, planned_grid_route, segment_clearance
from generate_test_environments import MAP_NODES

map_file = 'grid_map3.txt'
grid_map = np.loadtxt(map_file, dtype=int)
start, goal = MAP_NODES[Path(map_file).name][0], MAP_NODES[Path(map_file).name][-1]
geometry = dict(cell_size=(1., 1.), robot_radius=0., extra_margin=0.)


In [ ]:
raw_path, metadata = voronoi_astar_path(grid_map, start, goal, **geometry)
key_points, key_metadata = planned_grid_route(grid_map, start, goal, compress=True, **geometry)
print({'map_shape': grid_map.shape, 'raw': metadata, 'key_points': key_metadata})
if not key_points:
    raise RuntimeError('No safe route under the specified footprint and margin')
assert all(segment_clearance(grid_map, a, b, cell_size=geometry['cell_size']) >
           geometry['robot_radius'] + geometry['extra_margin']
           for a,b in zip(key_points, key_points[1:]))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(grid_map, cmap='gray_r', origin='upper')
if raw_path:
    points = np.asarray(raw_path)
    ax.plot(points[:,1], points[:,0], color='purple', label='Voronoi-A* (fallback labelled above)')
points = np.asarray(key_points)
ax.plot(points[:,1], points[:,0], 'r.-', label='Finite-segment key points')
ax.legend()
ax.set(xlabel='Column', ylabel='Row', title=map_file)
plt.show()
